In [1]:
from utils import load_manager
man_path = 'rediska0123/ue_manager_gsm8k_Qwen2.5-Math-7B'
scores_prm_path = 'configs/scores_prm_gsm8k_qwen1.5B.json'
scores_reasoneval_path = 'configs/scores_reasoneval_gsm8k_qwen1.5B.json'
man = load_manager(man_path)

In [2]:
import numpy as np

def parse_ans(s, ignore_unfinished=False):
    if '####' in s:
        return float(s.split('####')[-1].replace(',', ''))
    if r'\boxed{' in s:
        x = s.split(r'\boxed{')[-1].split('}')[0]
        return float(''.join(a for a in x if a.isdigit()))
    return None

def print_test_stats(man):
    stats = man['stats']
    acc = []
    for t, h in zip(stats['target_texts'], stats['greedy_texts']):
        at, ah = parse_ans(t), parse_ans(h)
        acc.append(np.isclose(at, ah) if ah is not None else 0)

    targets = man['gen_metrics']['claim', 'StepFactCheck']
    print('Total problems:', len(stats['input_texts']))
    print('Correct answers: {} ({}%)'.format(sum(acc), round(100 * np.mean(acc), 2)))
    print('Incorrect answers: {} ({}%)'.format(len(acc) - sum(acc), round(100 - 100 * np.mean(acc), 2)))
    print()
    print('Total steps: {}'.format(len(targets)))
    print('Correct steps: {} ({}%)'.format(len(targets) - sum(targets), round(100 - 100 * np.mean(targets), 2)))
    print('Incorrect steps: {} ({}%)'.format(sum(targets), round(100 * np.mean(targets), 2)))

print_test_stats(man)

Total problems: 500
Correct answers: 422 (84.4%)
Incorrect answers: 78 (15.6%)

Total steps: 2342
Correct steps: 2218 (94.71%)
Incorrect steps: 124 (5.29%)


In [3]:
import json
from utils import load_manager
from metrics import ROCAUC, PRAUC, ECE

def flatten(x):
    return [b for a in x for b in a]

estimations = man['estimations']
methods = {}
for (_, ue_name), ue_vals in estimations.items():
    methods[ue_name] = ue_vals

methods['Qwen2.5-Math-7B-PRM800K'] = [-y for y in flatten(json.load(open(scores_prm_path, 'r')))]
reasoneval_scores = flatten(json.load(open(scores_reasoneval_path, 'r')))
methods['ReasonEval'] = [s['redundancy'] - s['validity'] for s in reasoneval_scores]
methods['ReasonEval_validity'] = [-s['validity'] for s in reasoneval_scores]
methods['ReasonEval_redundancy'] = [s['redundancy'] for s in reasoneval_scores]

targets = man['gen_metrics']['claim', 'StepFactCheck']

for key, val in methods.items():
    print(f'{key}: {len(val)} values')

print(f'Targets: {len(targets)} values')

RandomBaselineClaim: 2342 values
MaximumClaimProbability: 2342 values
MaxTokenEntropyClaim: 2342 values
PerplexityClaim: 2342 values
LuqClaimEstimatorDummy_claim: 2342 values
Qwen2.5-Math-7B-PRM800K: 2342 values
ReasonEval: 2342 values
ReasonEval_validity: 2342 values
ReasonEval_redundancy: 2342 values
Targets: 2342 values


In [4]:
import pandas as pd
from metrics import ROCAUC, PRAUC, ECE
from plot_utils import pretty_plot_table
from collections import defaultdict

metrics = [ROCAUC(), PRAUC(), ECE()]
res_df = defaultdict(dict)

def rename_method(s):
    if s == 'RandomBaselineClaim':
        return 'Random'
    if s == 'MaximumClaimProbability':
        return 'MaxProb'
    if s == 'MaxTokenEntropyClaim':
        return 'MaxEntropy'
    if s == 'PerplexityClaim':
        return 'Perplexity'
    if 'LuqClaimEstimator' in s:
        return 'UHead'
    return s

for method_nm, method_vals in methods.items():
    if len(method_vals) != len(targets):
        print(f'Skipping {method_nm}: inconsistent number of samples, '
              f'expected {len(targets)}, got {len(method_vals)}')
        continue
    for m in metrics:
        res_df[str(m)][rename_method(method_nm)] = m(method_vals, targets)
    
df = pd.DataFrame(res_df)
pretty_plot_table(df)

,roc-auc,pr-auc,ece
Random,0.519361,0.057673,0.454584
MaxProb,0.671700,0.153856,0.669879
MaxEntropy,0.659080,0.100788,0.625870
Perplexity,0.670093,0.117352,0.053630
UHead,0.745324,0.188828,0.010841
Qwen2.5-Math-7B-PRM800K,0.843336,0.421020,0.044864
ReasonEval,0.820955,0.279997,0.044463
ReasonEval_validity,0.840353,0.306007,0.016984
ReasonEval_redundancy,0.636370,0.075132,0.047757
